> **Lösungsversion für Lehrkräfte.** Die Version für die Lernenden im Hauptordner wird daraus mit `python3 material/quelle/schuelerversion_erstellen.py` erzeugt: Code zwischen `# >>> LOESUNG` und `# <<< LOESUNG` wird entfernt, Zeilen mit `# | ` werden zum Platzhalter.

In [ ]:
# Nur in der Lehrkraft-Version: Hilfsmodul aus dem übergeordneten Ordner finden
import sys
sys.path.insert(0, "..")

# 11 – Vollvernetztes Netz (FCNN) trainieren und verbessern

**Lernsituation „KI-Sichtprüfstation“ · Doppelstunde 5**

In Notebook 10 hat das Netz gelernt – oft aber mit einer Lücke zwischen Trainings- und Validierungsgenauigkeit. Jetzt lernen Sie mit denselben Bildern Werkzeuge kennen, mit denen das Training besser wird. Alle Einstellungen stammen aus dem Original-Notebook des FCNN (`02a`).

**Ziele – am Ende können Sie …**
- Lernkurven lesen und Über- und Unteranpassung erkennen,
- Datenaugmentierung und EarlyStopping begründet einsetzen,
- verschiedene Netzgrößen vergleichen und das beste Modell auswählen,
- ein Modell mit Testdaten bewerten und speichern.

**So lesen Sie dieses Notebook**

| Kennzeichen | Bedeutung |
|---|---|
| **[Lesen]** | Grundlagen – lesen oder interaktiv ausprobieren |
| **[Aufgabe]** | Code ausführen, ergänzen oder selbst schreiben |
| **[Frage]** | Frage beantworten – schriftlich oder im Gespräch |
| **[Test]** | Teilzielkontrolle |
| **[Industrie]** | Bezug zur Fertigung |
| **[Raspberry Pi]** | Hinweis für den Raspberry Pi |

In [ ]:
# Werkzeuge laden – diese Zelle einfach ausführen (Umschalt + Enter)
import numpy as np
import cv2
import matplotlib.pyplot as plt
import ki_hilfen as kh

ORDNER = kh.ORDNER          # Ordner mit Notebooks, Datensätzen und Modellen

import keras
from keras import layers

kh.setze_seed(42)

## 1 · Einstellungen und Daten

`kh.lade_datensatz()` macht dasselbe wie Ihre Schleife aus Notebook 10 (laden, verkleinern, grau machen) und speichert das Ergebnis zusätzlich in einem Zwischenspeicher. Beim zweiten Mal geht das Laden deshalb sehr schnell.

In [ ]:
DATENSATZ = "dataset1"      # 176 Bilder – wie im Original-Notebook
GROESSE = 32                # Bilder werden auf 32 x 32 Pixel verkleinert
FARBE = "grau"              # Graustufen
BATCH = 32                  # Bilder pro Stapel
EPOCHEN = 500               # Obergrenze – EarlyStopping beendet meist früher
LERNRATE = 0.0003

X, y, klassen = kh.lade_datensatz(DATENSATZ, GROESSE, FARBE)
X_train, X_val, X_test, y_train, y_val, y_test = kh.teile_datensatz(X, y)
ANZAHL_KLASSEN = len(klassen)

print("Klassen:", klassen)
print("Training:", len(X_train), " Validierung:", len(X_val), " Test:", len(X_test))
kh.zeige_beispiele(X_train, y_train, klassen)

## 2 · [Lesen] Datenaugmentierung
Bei der **Augmentierung** wird jedes Trainingsbild in jeder Epoche zufällig ein wenig verändert: gespiegelt, gedreht, gezoomt. So sieht das Netz viel mehr Varianten und lernt weniger auswendig.

Die Augmentierung wird als erste Schicht in das Netz eingebaut. Sie arbeitet **nur beim Training**. Beim Testen und an der Prüfstation lässt sie die Bilder unverändert.

In [ ]:
augmentierung = keras.Sequential([
    layers.RandomFlip("horizontal"),         # links-rechts spiegeln
    layers.RandomRotation(0.1),              # bis zu ± 36° drehen
    layers.RandomZoom(0.05),                 # bis zu 5 % zoomen
])

kh.zeige_augmentierung(X_train[0], augmentierung)

### [Frage]
Ist Spiegeln für unsere Symbole sinnvoll? Und für ein Bauteil, bei dem geprüft werden soll, ob es **richtig herum** liegt?

[Industrie] Augmentierung darf nie das verändern, was die Klasse ausmacht.

In [ ]:
train_ds = kh.als_tf_dataset(X_train, y_train, BATCH, mischen=True)   # Stapel zu 32 Bildern, gemischt
val_ds = kh.als_tf_dataset(X_val, y_val, BATCH)
test_ds = kh.als_tf_dataset(X_test, y_test, BATCH)

## 3 · Drei Netzgrößen

Wir vergleichen drei vollvernetzte Netze: **small**, **medium** und **big**. Alle beginnen mit der Augmentierung, der Normierung (`Rescaling`) und `Flatten` und enden mit einer Softmax-Schicht für die Klassen.

| Variante | Schichten (Neuronen) | Besonderheit |
|---|---|---|
| small | 32 → 128 → 64 → 32 | nach jeder Schicht `Dropout(0.3)` |
| medium | 64 → 256 → 128 → 64 | ohne Dropout |
| big | 64 → 1024 → 256 → 64 | ohne Dropout, sehr viele Gewichte |

### [Aufgabe 1] Die Variante „medium“ ergänzen
Die Variante „small“ ist vorgegeben. Ergänzen Sie „medium“ nach der Tabelle: vier `Dense`-Schichten mit `activation="relu"`, ohne Dropout.

In [ ]:
def baue_fcnn(variante="medium", aug=True):
    if variante == "small":
        schichten = [
            layers.Dense(32, activation="relu"), layers.Dropout(0.3),
            layers.Dense(128, activation="relu"), layers.Dropout(0.3),
            layers.Dense(64, activation="relu"), layers.Dropout(0.3),
            layers.Dense(32, activation="relu"), layers.Dropout(0.3),
        ]
    elif variante == "medium":
        # >>> LOESUNG
        # | schichten = []   # TODO: vier Dense-Schichten nach der Tabelle
        schichten = [
            layers.Dense(64, activation="relu"),
            layers.Dense(256, activation="relu"),
            layers.Dense(128, activation="relu"),
            layers.Dense(64, activation="relu"),
        ]
        # <<< LOESUNG
    else:   # "big"
        schichten = [
            layers.Dense(64, activation="relu"),
            layers.Dense(1024, activation="relu"),
            layers.Dense(256, activation="relu"),
            layers.Dense(64, activation="relu"),
        ]
    vorne = [keras.Input(shape=X_train.shape[1:])]
    if aug:
        vorne.append(augmentierung)                   # nur beim Training aktiv
    vorne += [layers.Rescaling(1 / 255),              # 0 ... 255  ->  0 ... 1
              layers.Flatten()]
    return keras.Sequential(vorne + schichten + [layers.Dense(ANZAHL_KLASSEN, activation="softmax")])


modell = baue_fcnn("medium")
modell.summary()
kh.pruefe_gleich(modell.count_params(), 123652, "Gewichte der Variante medium")

[Lesen] `layers.Rescaling(1 / 255)` teilt jeden Pixelwert durch 255 – genau wie Ihre Normierung in Notebook 10, nur jetzt als Schicht im Netz. Ohne diese Normierung lernt das Netz nicht. Weil sie im Netz steckt, bekommt es auch an der Prüfstation automatisch dieselben Werte wie beim Training.

## 4 · Training mit sinkender Lernrate und EarlyStopping

| Werkzeug | Aufgabe |
|---|---|
| `ExponentialDecay` | Lernrate wird mit der Zeit kleiner: am Anfang große Schritte, später feine |
| `EarlyStopping` | beendet das Training, wenn der Validierungsverlust 50 Epochen lang nicht besser wird |
| `ModelCheckpoint` | speichert die Gewichte der **besten** Epoche – diese laden wir danach |

In [ ]:
def trainiere(modell, lernrate=LERNRATE, epochen=EPOCHEN, name="fcnn"):
    lernplan = keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=lernrate, decay_steps=500, decay_rate=0.9)
    modell.compile(optimizer=keras.optimizers.Adam(lernplan),
                   loss="sparse_categorical_crossentropy",   # wie categorical, nur ohne One-Hot
                   metrics=["accuracy"])
    bestes = kh.modellpfad(name + "_bestes.weights.h5")
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=50),
        keras.callbacks.ModelCheckpoint(bestes, monitor="val_loss",
                                        save_best_only=True, save_weights_only=True),
    ]
    verlauf = modell.fit(train_ds, validation_data=val_ds, epochs=epochen,
                         callbacks=callbacks, verbose=0)
    modell.load_weights(bestes)
    beste = int(np.argmin(verlauf.history["val_loss"]))
    print("Trainiert:", len(verlauf.epoch), "Epochen · beste Epoche:", beste + 1,
          "· Val-Genauigkeit dort:", round(verlauf.history["val_accuracy"][beste], 3))
    return verlauf


print("Training läuft – bitte warten ...")
verlauf = trainiere(modell)
kh.zeige_lernkurven(verlauf)

## 5 · [Lesen] Lernkurven deuten
Nutzen Sie den Simulator und das Diagnose-Training. Deuten Sie danach Ihre eigenen Lernkurven von oben.

In [ ]:
kh.zeige_html("05_lernkurven.html")

## 6 · [Aufgabe 2] Netzgrößen vergleichen
Jede Gruppe trainiert eine Variante (oder zwei) und trägt das Ergebnis in die Tabelle ein. Die Funktion `experiment()` macht alles in einem Schritt.

| Gruppe | Versuch |
|---|---|
| 1 | `experiment("small")` |
| 2 | `experiment("medium")` |
| 3 | `experiment("big")` |
| 4 | `experiment("medium", aug=False)` – ohne Augmentierung |

[Raspberry Pi] Ein Versuch dauert etwa 1 bis 3 Minuten.

In [ ]:
ergebnisse = []

def experiment(variante, lernrate=LERNRATE, aug=True):
    m = baue_fcnn(variante, aug=aug)
    v = trainiere(m, lernrate=lernrate, name="experiment")
    _, test_acc = m.evaluate(test_ds, verbose=0)
    ergebnisse.append({"Variante": variante, "Augmentierung": aug, "Lernrate": lernrate,
                       "Gewichte": m.count_params(), "Epochen": len(v.epoch),
                       "Train-Genauigkeit": round(max(v.history["accuracy"]), 3),
                       "Val-Genauigkeit": round(max(v.history["val_accuracy"]), 3),
                       "Test-Genauigkeit": round(test_acc, 3)})
    kh.zeige_lernkurven(v)
    return kh.tabelle(ergebnisse)


experiment("small")

### [Frage]
1. Welche Variante war am besten? Lohnt sich das große Netz?
2. Woran erkennen Sie in der Tabelle Überanpassung? (Tipp: Train- und Val-Genauigkeit vergleichen.)
3. Die Variante „small“ bleibt oft in der Nähe von 25 % (Raten) und wird früh gestoppt. Woran könnte das liegen? (Tipp: Dropout 0,3 bei Schichten mit nur 32 Neuronen – Unteranpassung.)
4. Zusatz: Starten Sie `experiment("medium", lernrate=0.01)`. Was passiert mit den Lernkurven?

## 7 · Bewertung mit den Testdaten

In [ ]:
_, test_acc = modell.evaluate(test_ds, verbose=0)
print("Test-Genauigkeit:", round(test_acc * 100, 1), "%")

y_pred = modell.predict(X_test, verbose=0).argmax(axis=1)
kh.zeige_konfusionsmatrix(y_test, y_pred, klassen, titel="FCNN – Testdaten")

### [Aufgabe 3] Falsch erkannte Bilder anzeigen
Finden Sie die Nummern aller Testbilder, bei denen `y_pred` nicht mit `y_test` übereinstimmt. Hinweis: `np.where(y_pred != y_test)[0]`.

In [ ]:
# >>> LOESUNG
# | falsch = None   # TODO
falsch = np.where(y_pred != y_test)[0]
# <<< LOESUNG

print(len(falsch), "von", len(y_test), "Testbildern falsch erkannt")
kh.pruefe_gleich(len(falsch), int((y_pred != y_test).sum()), "Anzahl gefundener Fehler")
kh.zeige_beispiele(X_test[falsch], y_test[falsch], klassen, vorhersage=y_pred[falsch])

### [Frage]
Was haben die falsch erkannten Bilder gemeinsam (Licht, Lage, Strichstärke, Bildausschnitt)? Welche Bilder sollten Sie zusätzlich aufnehmen?

## 8 · Modell speichern
Gespeichert werden das Modell (`.keras`), eine schnelle Version für den Raspberry Pi (`.tflite`) und eine Infodatei mit Klassen und Bildgröße.

In [ ]:
kh.speichere_modell(modell, "fcnn", klassen, GROESSE, FARBE,
                    test_genauigkeit=round(float(test_acc), 4), datensatz=DATENSATZ)

## [Test] Teilzielkontrolle 2

In [ ]:
kh.zeige_html("tzk2_neuronale_netze.html")

### Ausblick
Das FCNN betrachtet jedes Pixel einzeln. Liegt ein Symbol ein paar Pixel weiter rechts, sieht das Netz ganz andere Zahlen. **Faltungsnetze (CNN)** in Notebook 12 nutzen dagegen die Nachbarschaft der Pixel.

In [ ]:
kh.ram_aufraeumen()